# Dropout

在神经网络的训练过程中，它会以一定的概率 $p$（超参数，通常设为 0.5）随机让一部分神经元停止工作,通过“随机罢工”的方式，强迫模型变得更加强壮

需要满足： $$\mathbb{E}[x'] = x$$

因为
$$\mathbb{E}[x'] = p \cdot 0 + (1-p) \cdot \frac{x_i}{1-p}$$
所以 
$$\mathbb{E}[x'] = x$$

只在训练时使用，推理时不需要 Dropout

## Dropout 实现

In [ ]:
import torch
from torch import nn

def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1
    # 如果 dropout 为 1，全部丢弃
    if dropout == 1:
        return torch.zeros_like(X)
    # 如果 dropout 为 0，全部保留
    if dropout == 0:
        return X
    
    # 生成掩码 (mask)：以 (1-p) 的概率生成 1
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 定义包含 Dropout 的模型
class DropoutMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, p1, p2):
        super(DropoutMLP, self).__init__()
        self.lin1 = nn.Linear(input_size, hidden_size)
        self.lin2 = nn.Linear(hidden_size, hidden_size)
        self.lin3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
        
        # 使用 PyTorch 内置的 Dropout，它已经实现了你图片中的缩放逻辑
        self.dropout1 = nn.Dropout(p1)
        self.dropout2 = nn.Dropout(p2)

    def forward(self, x):
        # 第一层后加 Dropout
        x = self.relu(self.lin1(x))
        x = self.dropout1(x)
        
        # 第二层后加 Dropout
        x = self.relu(self.lin2(x))
        x = self.dropout2(x)
        
        x = self.lin3(x)
        return x

# 2. 准备模拟数据
X_train = torch.randn(100, 20)
Y_train = torch.randn(100, 1)

# 3. 实例化模型 (设置两个隐藏层的丢弃概率)
model = DropoutMLP(input_size=20, hidden_size=64, output_size=1, p1=0.2, p2=0.5)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# 4. 训练模式
model.train() # 训练阶段，开启 Dropout
for epoch in range(5):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, Y_train)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 5. 测试模式
model.eval()  # 预测阶段，关闭 Dropout，使用所有神经元
with torch.no_grad():
    test_input = torch.randn(1, 20)
    prediction = model(test_input)
    print("\n测试预测值:", prediction.item())